# Deploy SourceExecutor to Avalanche

This notebook deploys `SourceExecutor.sol` from the current folder to Avalanche C-Chain.

## Prereqs
- Use the `myvenv` kernel.
- Export the environment variable:
  - `DEPLOYER_PRIVATE_KEY`
- Set RPC/odos/usdc/token_messenger addresses directly in the notebook.
- The deployer wallet must have AVAX for gas.


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv
from solcx import compile_standard, install_solc
from web3 import Web3

load_dotenv(Path("..") / ".env")

CHAIN_ID = 43114

AVALANCHE_RPC_URL = "https://api.avax.network/ext/bc/C/rpc"
ODOS_ROUTER_ADDRESS = "0x0D05a7D3448512B78fa8A9e46c4872C88C4a0D05"
USDC_ADDRESS = "0xB97EF9Ef8734C71904D8002F8b6Bc66Dd9c48a6E"
TOKEN_MESSENGER_ADDRESS = "0x28b5a0e9C621a5BadaA536219b3a228C8168cf5d"


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Missing env var: {name}")
    return value


def require_address(label: str, value: str) -> str:
    if not value or value == "0x0000000000000000000000000000000000000000":
        raise ValueError(f"Set {label} address")
    return Web3.to_checksum_address(value)


rpc_url = AVALANCHE_RPC_URL
private_key = require_env("DEPLOYER_PRIVATE_KEY")
odos_router = require_address("ODOS_ROUTER", ODOS_ROUTER_ADDRESS)
usdc_address = require_address("USDC_ADDRESS", USDC_ADDRESS)
token_messenger = require_address("TOKEN_MESSENGER", TOKEN_MESSENGER_ADDRESS)

In [6]:
w3 = Web3(Web3.HTTPProvider(rpc_url))
if not w3.is_connected():
    raise ConnectionError("Failed to connect to RPC")

install_solc("0.8.19")
source = Path("SourceExecutor.sol").read_text()

compiled = compile_standard(
    {
        "language": "Solidity",
        "sources": {"SourceExecutor.sol": {"content": source}},
        "settings": {"outputSelection": {"*": {"*": ["abi", "evm.bytecode"]}}},
    },
    solc_version="0.8.19",
)

contract_data = compiled["contracts"]["SourceExecutor.sol"]["SourceExecutor"]
abi = contract_data["abi"]
bytecode = contract_data["evm"]["bytecode"]["object"]


In [8]:
account = w3.eth.account.from_key(private_key)
print("Deployer:", account.address)

contract = w3.eth.contract(abi=abi, bytecode=bytecode)

constructor = contract.constructor(odos_router, usdc_address, token_messenger)

gas_estimate = constructor.estimate_gas({"from": account.address})

tx = constructor.build_transaction(
    {
        "from": account.address,
        "nonce": w3.eth.get_transaction_count(account.address),
        "gas": int(gas_estimate * 1.2),
        "gasPrice": w3.eth.gas_price,
        "chainId": CHAIN_ID,
    }
)

signed = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
print("Tx hash:", tx_hash.hex())

receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Contract address:", receipt.contractAddress)


Deployer: 0x2299fFB140C4f437E0015fdd9e1956feb2471eA6
Tx hash: 4ff35cacb2c7b068cb2d89f6cca8fe29dc8d20ae584762ea6363c5fbc1e3ac28
Contract address: 0xeC20cbb6C99e4b8b78cbd81b27Ce0B7aaD222A2d
